<a href="https://colab.research.google.com/github/arturbernardo/word_vectors/blob/main/llm_vectors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install transformers torch sentencepiece

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
MODEL_NAME = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

embedding_matrix = model.get_input_embeddings().weight.detach()

print("Shape da matriz de embeddings:", tuple(embedding_matrix.shape))
print("Tamanho do vocabulário:", embedding_matrix.shape[0])
print("Dimensão dos embeddings:", embedding_matrix.shape[1])

In [4]:
def nearest_to_vector(vec, topn=10):
    vec = vec / vec.norm()
    emb_norm = embedding_matrix / embedding_matrix.norm(dim=1, keepdim=True)
    sims = torch.mv(emb_norm, vec)

    top_ids = torch.topk(sims, k=topn).indices.tolist()

    for idx in top_ids:
        tok = tokenizer.convert_ids_to_tokens([idx])[0]
        print(tok, float(sims[idx]))

def get_embedding(token_text):
    ids = tokenizer.encode(token_text, add_special_tokens=False)

    if len(ids) != 1:
        raise ValueError(f"'{token_text}' virou vários tokens: {tokenizer.convert_ids_to_tokens(ids)}")

    return embedding_matrix[ids[0]]

# Proximidade de palavras

eval de (king - man + woman) = queen com 0.70 de peso

eval de (king) = queen com 0.66 de peso

Distanciar mais king de man da um resultado mais próximo de rainha,
mostrando a relação geométrica e como ela acaba revelando significados.

In [5]:
king = get_embedding(" king")
man = get_embedding(" man")
woman = get_embedding(" woman")

# analogia vetorial
v = king - man + woman

# buscar tokens mais próximos
nearest_to_vector(v, topn=15)

Ġking 0.7512587308883667
Ġqueen 0.7097044587135315
Ġprincess 0.6059992909431458
ĠQueen 0.5824058055877686
Ġkings 0.5818101167678833
Queen 0.5543056726455688
Ġwoman 0.5230833292007446
Ġmonarch 0.5188010334968567
Ġqueens 0.5171242952346802
ĠKing 0.5156242251396179
Ġgoddess 0.5097447633743286
Ġruler 0.5057487487792969
Ġheroine 0.5039258003234863
ĠEmpress 0.4975624084472656
Ġroyal 0.4935463070869446


In [6]:
nearest_to_vector(king, topn=15)

Ġking 1.0
Ġkings 0.7347280979156494
ĠKing 0.6768434047698975
Ġqueen 0.6639549732208252
King 0.6070865392684937
Ġprince 0.6012954711914062
Ġmonarch 0.5855213403701782
Ġruler 0.5693265199661255
Ġemperor 0.5625233054161072
Ġroyal 0.5464542508125305
Ġprinces 0.5378309488296509
ĠQueen 0.5332906246185303
Ġkingdom 0.5309536457061768
Ġlord 0.5297277569770813
Ġthrone 0.5278258323669434


# Próximo token de uma frase

In [7]:
text = "The king lives in the"
inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

next_token_logits = logits[0, -1, :]

probs = torch.softmax(next_token_logits, dim=-1)

next_token_id = torch.argmax(probs).item()
next_token = tokenizer.decode([next_token_id])

print("Frase:", text)
print("Próximo token mais provável:", repr(next_token))
print("Probabilidade:", probs[next_token_id].item())

Frase: The king lives in the
Próximo token mais provável: ' heart'
Probabilidade: 0.03182327002286911


In [8]:
top_k = 10
top_probs, top_ids = torch.topk(probs, k=top_k)

print(f"Top {top_k} próximos tokens:")
for rank, (token_id, prob) in enumerate(zip(top_ids.tolist(), top_probs.tolist()), start=1):
    token_str = tokenizer.decode([token_id])
    print(f"{rank:2d}. {repr(token_str):15s}  prob={prob:.6f}")

Top 10 próximos tokens:
 1. ' heart'         prob=0.031823
 2. ' same'          prob=0.019507
 3. ' shadow'        prob=0.018451
 4. ' middle'        prob=0.018428
 5. ' land'          prob=0.018106
 6. ' city'          prob=0.017504
 7. ' palace'        prob=0.016767
 8. ' mountains'     prob=0.016527
 9. ' forest'        prob=0.016222
10. ' woods'         prob=0.015447


# Proximidade de vetores resultantes de textos

Versão de (king - man + woman) usando textos, já que LLMs são mais "favoráveis" a contextos criados desta forma.

In [ ]:
!pip -q install sentence-transformers

from sentence_transformers import SentenceTransformer
import numpy as np


In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

# retorna vetor a partir de um texto
def emb(text):
    return model.encode(text)

def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def nearest(target, candidates):
    scores = [(c, cosine(target, emb(c))) for c in candidates]
    return sorted(scores, key=lambda x: -x[1])

a = emb("A king is a powerful man")
b = emb("A man")
c = emb("A woman")

v = a - b + c

candidates = [
    "A queen is a powerful woman",
    "A king is a powerful man",
    "A princess",
    "A female leader",
    "A king",
    "A man",
]

results = nearest(v, candidates)

for text, score in results:
    print(f"{score:.4f} - {text}")


# o própria frase original, "A king is a powerful man",
# ficou mais distante que a frase com queen.

# 0.7543 - A queen is a powerful woman
# 0.5065 - A king is a powerful man
# 0.4844 - A female leader
# 0.4624 - A king
# 0.4514 - A princess
# -0.2551 - A man